# CS224W — LightGCN on all Electronics users

Open in Colab and select a high-memory GPU runtime if available. The default uses 16-dimensional embeddings to reduce GPU memory. If a fresh 15 GiB runtime still runs out of memory, try 8 dimensions or use a GPU with more memory. Allow substantial local disk space for the raw reviews, DuckDB tables, disk-backed CSR arrays, and spill files. Full-data mode requires a CUDA GPU in this notebook. Toy mode also works on CPU.

Set `USE_TOY_DATA = True` to smoke-test the same pipeline without downloading reviews. Set it to False and run all cells for Electronics. The real run includes all valid users before the temporal split. The training graph contains all training edges. BPR samples positive training edges per step, and validation ranks the full training catalog for a deterministic sample of up to 10,000 warm-start users. Test data is reserved for later.

The validation cohort and catalog differ from the 1% pilot, so compare LightGCN and Most Popular within this run.

In [ ]:
%pip -q install duckdb huggingface_hub pyarrow

In [ ]:
import shutil
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from torch import nn
from torch.nn import functional as F

USE_TOY_DATA = False  # Set True for a quick check without downloading reviews.
SEED = 42
EVAL_CUTOFFS = (20, 50, 100)
MAX_VALIDATION_USERS = 10_000
DUCKDB_MEMORY_LIMIT = '1GB'
DUCKDB_THREADS = 1
EMBEDDING_DIM = 16  # Try 8 if a fresh GPU runtime still runs out of memory.
NUM_LAYERS = 2
EPOCHS = 5
STEPS_PER_EPOCH = 20
TRAIN_BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 32

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif not USE_TOY_DATA:
    raise RuntimeError("Select a GPU runtime in Colab before running full Electronics.")

## 1. Build chronological splits for all users

In [ ]:
workdir = Path("lightgcn_toy_work" if USE_TOY_DATA else "lightgcn_full_work")
workdir.mkdir(exist_ok=True)
con = duckdb.connect(str(workdir / "baseline.duckdb"))
spill_dir = workdir / "duckdb_spill"
spill_dir.mkdir(exist_ok=True)
con.execute("SET temp_directory = ?", [str(spill_dir.resolve())])
con.execute("SET memory_limit = ?", [DUCKDB_MEMORY_LIMIT])
con.execute("SET threads = ?", [DUCKDB_THREADS])
con.execute("SET preserve_insertion_order = false")
print(f"Free disk: {shutil.disk_usage(workdir).free / 1e9:.1f} GB", flush=True)

if USE_TOY_DATA:
    # Toy timestamps create repeat users and warm items in validation.
    con.execute("""
        CREATE OR REPLACE TABLE interactions AS
        SELECT 'u' || u AS user_id,
               'i' || ((u + j) % 80) AS parent_asin,
               100 * j + u % 11 AS first_timestamp
        FROM range(120) AS users(u)
        CROSS JOIN range(10) AS offsets(j)
    """)
else:
    print("Downloading or locating raw Electronics reviews...", flush=True)
    review_path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        repo_type="dataset",
        filename="raw/review_categories/Electronics.jsonl",
    )
    print("Deduplicating full Electronics reviews...", flush=True)
    con.execute("""
        CREATE OR REPLACE TABLE interactions AS
        SELECT user_id, parent_asin, MIN(timestamp) AS first_timestamp
        FROM read_json(
            ?, format='newline_delimited',
            columns={
                'user_id': 'VARCHAR',
                'parent_asin': 'VARCHAR',
                'timestamp': 'BIGINT'
            }
        )
        WHERE user_id IS NOT NULL
          AND parent_asin IS NOT NULL
          AND timestamp IS NOT NULL
            GROUP BY user_id, parent_asin
    """, [review_path])

print("Computing chronological split...", flush=True)
val_start, test_start = con.execute("""
    SELECT quantile_disc(first_timestamp, 0.80),
           quantile_disc(first_timestamp, 0.90)
    FROM interactions
""").fetchone()

if val_start is None or val_start >= test_start:
    raise ValueError("Need more interactions with distinct timestamps.")

con.execute("""
    CREATE OR REPLACE TABLE baseline_split AS
    SELECT *, CASE WHEN first_timestamp < ? THEN 'train'
                   WHEN first_timestamp < ? THEN 'validation'
                   ELSE 'test' END AS split
    FROM interactions
""", [val_start, test_start])

display(con.execute("""
    SELECT split, COUNT(*) AS interactions,
           COUNT(DISTINCT user_id) AS users,
           COUNT(DISTINCT parent_asin) AS items
    FROM baseline_split
    GROUP BY split ORDER BY MIN(first_timestamp)
""").df())

## 2. Index and build the full training graph

DuckDB maps string IDs to integers. The symmetric graph uses CSR with 32-bit indices and float32 weights. CPU history lookups use slices of the same CSR arrays, so there is no Python set per training user.
DuckDB spills to local disk with a smaller memory budget. The degree and edge arrays are backed by local files.


In [ ]:
print("Indexing training users and items...", flush=True)
con.execute("""
    CREATE OR REPLACE TABLE train_users AS
    SELECT user_id, (ROW_NUMBER() OVER (ORDER BY user_id) - 1)::INTEGER AS uid
    FROM (SELECT DISTINCT user_id FROM baseline_split WHERE split = 'train')
""")
con.execute("""
    CREATE OR REPLACE TABLE train_items AS
    SELECT parent_asin, (ROW_NUMBER() OVER (ORDER BY parent_asin) - 1)::INTEGER AS iid
    FROM (SELECT DISTINCT parent_asin FROM baseline_split WHERE split = 'train')
""")
con.execute("""
    CREATE OR REPLACE TABLE train_edges AS
    SELECT uid, iid
    FROM baseline_split
    JOIN train_users USING (user_id)
    JOIN train_items USING (parent_asin)
    WHERE split = 'train'
""")

n_users = con.execute("SELECT COUNT(*) FROM train_users").fetchone()[0]
n_items = con.execute("SELECT COUNT(*) FROM train_items").fetchone()[0]
n_edges = con.execute("SELECT COUNT(*) FROM train_edges").fetchone()[0]
n_nodes = n_users + n_items
if n_items < 2 or not n_edges:
    raise ValueError("Need training edges and at least two training items.")
if max(n_nodes, 2 * n_edges) > np.iinfo(np.int32).max:
    raise ValueError("Graph exceeds the 32-bit CSR index limit.")
print(f"Training graph: {n_users:,} users, {n_items:,} items, {n_edges:,} edges")
print(f"One float32 embedding matrix: {n_nodes * EMBEDDING_DIM * 4 / 1e9:.2f} GB")

def disk_array(name, size, dtype):
    return np.lib.format.open_memmap(
        workdir / f"{name}.npy", mode="w+", dtype=dtype, shape=(size,)
    )


print("Building CSR degree offsets...", flush=True)
degrees = disk_array("degrees", n_nodes, np.int32)
for node_id, start_node in (("uid", 0), ("iid", n_users)):
    count_batches = con.execute(f"""
        SELECT {node_id}, COUNT(*)::INTEGER AS degree
        FROM train_edges GROUP BY {node_id}
    """).fetch_record_batch(100_000)
    for batch in count_batches:
        nodes = batch.column(0).to_numpy() + start_node
        degrees[nodes] = batch.column(1).to_numpy()
degrees.flush()

item_frequency = degrees[n_users:]
row_ptr = disk_array("row_ptr", n_nodes + 1, np.int32)
row_ptr[0] = 0
np.cumsum(degrees, dtype=np.int32, out=row_ptr[1:])
row_ptr.flush()

columns = disk_array("columns", 2 * n_edges, np.int32)
weights = disk_array("weights", 2 * n_edges, np.float32)
print("Sorting and streaming normalized edges to disk...", flush=True)
offset = 0
reader = con.execute("""
    SELECT uid AS src, (iid + ?)::INTEGER AS dst FROM train_edges
    UNION ALL
    SELECT (iid + ?)::INTEGER AS src, uid AS dst FROM train_edges
    ORDER BY src, dst
""", [n_users, n_users]).fetch_record_batch(100_000)
for batch in reader:
    src = batch.column(0).to_numpy()
    dst = batch.column(1).to_numpy()
    end = offset + len(dst)
    columns[offset:end] = dst
    weights[offset:end] = 1 / np.sqrt(
        degrees[src].astype(np.float32) * degrees[dst].astype(np.float32)
    )
    offset = end
assert offset == 2 * n_edges
columns.flush()
weights.flush()
print("Transferring CSR graph to the training device...", flush=True)
adjacency = torch.sparse_csr_tensor(
    torch.from_numpy(row_ptr).to(device),
    torch.from_numpy(columns).to(device),
    torch.from_numpy(weights).to(device),
    size=(n_nodes, n_nodes),
)
del weights

## 3. Select warm-start validation users

Keep validation pairs whose user and target item both appear in training. Select users by stable hash and keep every eligible target for each selected user. Report coverage before sampling.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE warm_validation AS
    SELECT uid, iid
    FROM baseline_split
    JOIN train_users USING (user_id)
    JOIN train_items USING (parent_asin)
    WHERE split = 'validation'
""")
warm_pairs = con.execute("SELECT COUNT(*) FROM warm_validation").fetchone()[0]
total_validation_pairs = con.execute(
    "SELECT COUNT(*) FROM baseline_split WHERE split = 'validation'"
).fetchone()[0]
validation_rows = con.execute("""
    WITH selected AS (
        SELECT uid
        FROM (SELECT DISTINCT uid FROM warm_validation)
        JOIN train_users USING (uid)
        ORDER BY md5_number_lower(user_id), uid
        LIMIT ?
    )
    SELECT uid, iid FROM warm_validation JOIN selected USING (uid)
    ORDER BY uid, iid
""", [MAX_VALIDATION_USERS]).fetchall()
positives = {}
for u, i in validation_rows:
    positives.setdefault(u, set()).add(i)
if not positives:
    raise ValueError("No warm-start validation pairs.")
print(f"Warm-start validation: {warm_pairs:,}/{total_validation_pairs:,} pairs")
print(f"Evaluating {len(positives):,} users and {len(validation_rows):,} target pairs")
# Release DuckDB buffers before scoring and training.
con.close()


## 4. Shared full-catalog evaluation

Both models mask training interactions and rank the same catalog. Metrics are averaged across users with eligible targets. The low-interaction group keeps the same ranked lists and counts only targets with 1–5 training interactions.

In [ ]:
def evaluate(model_name, score_batch):
    groups = ("overall", "sparse_items_1_to_5")
    totals = np.zeros((len(groups), len(EVAL_CUTOFFS), 2))
    counts = np.zeros(len(groups), dtype=np.int64)
    users = list(positives)
    max_k = min(max(EVAL_CUTOFFS), n_items)
    discount = 1 / np.log2(np.arange(max_k) + 2)
    ideal = np.concatenate(([0.0], np.cumsum(discount)))

    for start in range(0, len(users), EVAL_BATCH_SIZE):
        batch = users[start:start + EVAL_BATCH_SIZE]
        scores = score_batch(batch)
        seen = [columns[row_ptr[u]:row_ptr[u + 1]] - n_users for u in batch]
        rows = np.repeat(np.arange(len(batch)), [len(items) for items in seen])
        scores[torch.as_tensor(rows, device=device),
               torch.as_tensor(np.concatenate(seen), dtype=torch.long, device=device)] = -torch.inf
        values, indices = scores.topk(max_k, dim=1)
        for u, ranked, valid in zip(batch, indices.tolist(), values.isfinite().tolist()):
            relevant = positives[u]
            sparse = {i for i in relevant if item_frequency[i] <= 5}
            for group_index, targets in enumerate((relevant, sparse)):
                if not targets:
                    continue
                counts[group_index] += 1
                hits = np.array([available and i in targets for i, available in zip(ranked, valid)])
                recall_prefix = np.cumsum(hits)
                dcg_prefix = np.cumsum(hits * discount)
                for cutoff_index, k in enumerate(EVAL_CUTOFFS):
                    depth = min(k, max_k)
                    totals[group_index, cutoff_index] += (
                        recall_prefix[depth - 1] / len(targets),
                        dcg_prefix[depth - 1] / ideal[min(depth, len(targets))],
                    )

    rows = []
    for group_index, group in enumerate(groups):
        count = counts[group_index]
        row = {"model": model_name, "group": group, "users": count}
        for cutoff_index, k in enumerate(EVAL_CUTOFFS):
            recall, ndcg = totals[group_index, cutoff_index] / count if count else (np.nan, np.nan)
            row.update({f"Recall@{k}": recall, f"NDCG@{k}": ndcg})
        rows.append(row)
    return pd.DataFrame(rows)

popular_scores = torch.as_tensor(item_frequency, dtype=torch.float64, device=device)
popular_scores -= torch.arange(n_items, dtype=torch.float64, device=device) / n_items
popular_results = evaluate(
    "Most Popular", lambda users: popular_scores.expand(len(users), -1).clone()
)
display(popular_results)

## 5. Train LightGCN

Each optimizer step propagates over the full training graph and optimizes BPR on a sampled batch of training edges. The default budget is 100 optimizer steps (5 reporting intervals of 20 steps each). An interval does not represent a pass through all training edges. Increase the step budget for a longer run.

The fixed normalized graph is symmetric, so backpropagation through graph multiplication can reuse the same CSR matrix.

In [ ]:
class SymmetricGraphMultiply(torch.autograd.Function):
    @staticmethod
    def forward(ctx, graph, embeddings):
        ctx.save_for_backward(graph)
        return torch.sparse.mm(graph, embeddings)

    @staticmethod
    def backward(ctx, grad_output):
        (graph,) = ctx.saved_tensors
        return None, torch.sparse.mm(graph, grad_output.contiguous())


class LightGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(n_nodes, EMBEDDING_DIM)
        nn.init.normal_(self.embedding.weight, std=0.1)

    def forward(self):
        current = self.embedding.weight
        total = current
        for _ in range(NUM_LAYERS):
            current = SymmetricGraphMultiply.apply(adjacency, current)
            total = total + current
        return total / (NUM_LAYERS + 1)


if not np.any(degrees[:n_users] < n_items):
    raise ValueError("No training user has an eligible negative item.")

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

model = LightGCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, foreach=False)
user_ptr = row_ptr[:n_users + 1]
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for _ in range(STEPS_PER_EPOCH):
        sample = rng.integers(n_edges, size=min(TRAIN_BATCH_SIZE, n_edges))
        users = np.searchsorted(user_ptr, sample, side="right") - 1
        eligible = degrees[users] < n_items
        sample, users = sample[eligible], users[eligible]
        if not len(sample):
            continue

        negative_nodes = rng.integers(n_users, n_nodes, size=len(sample))
        for j, user in enumerate(users):
            seen = columns[user_ptr[user]:user_ptr[user + 1]]
            while True:
                idx = np.searchsorted(seen, negative_nodes[j])
                if idx == len(seen) or seen[idx] != negative_nodes[j]:
                    break
                negative_nodes[j] = rng.integers(n_users, n_nodes)

        u = torch.as_tensor(users, device=device)
        p = torch.as_tensor(columns[sample], dtype=torch.long, device=device)
        n = torch.as_tensor(negative_nodes, device=device)
        optimizer.zero_grad(set_to_none=True)
        embeddings = model()
        margins = (embeddings[u] * (embeddings[p] - embeddings[n])).sum(dim=1)
        loss = F.softplus(-margins).mean()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        del embeddings, loss, margins

    print(f"Interval {epoch + 1}: BPR loss = {epoch_loss / STEPS_PER_EPOCH:.4f}", flush=True)
optimizer.zero_grad(set_to_none=True)
del optimizer
if device.type == "cuda":
    print(f"Peak GPU tensor memory during training: {torch.cuda.max_memory_allocated(device) / 2**30:.2f} GiB")

## 6. Export validation results

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model()
    user_embeddings = embeddings[:n_users]
    item_embeddings = embeddings[n_users:]
    lightgcn_results = evaluate(
        "LightGCN", lambda users: user_embeddings[users] @ item_embeddings.T
    )

results = pd.concat([popular_results, lightgcn_results], ignore_index=True)
results.insert(0, "dataset", "toy" if USE_TOY_DATA else "Electronics")
results["lightgcn_embedding_dim"] = EMBEDDING_DIM
results["train_users"] = n_users
results["train_items"] = n_items
results["train_pairs"] = n_edges
results["warm_validation_pairs"] = warm_pairs
results["total_validation_pairs"] = total_validation_pairs
results["sampled_validation_pairs"] = len(validation_rows)
display(results)
output = workdir / ("toy_validation.csv" if USE_TOY_DATA else "electronics_full_validation.csv")
results.to_csv(output, index=False)
print("Saved:", output)